# 05 — Métricas y Comparación

Cuadro completo de métricas sobre el conjunto de test.


In [ ]:
import sys
from pathlib import Path

_bootstrap = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(_bootstrap / 'src') not in sys.path:
    sys.path.insert(0, str(_bootstrap / 'src'))

from discurso_odio.paths import PROJECT_ROOT

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from discurso_odio.config import load_settings
from discurso_odio.data.loaders import load_all_splits, load_split, get_class_weights
from discurso_odio.paths import REPORTS_FIGURES, REPORTS_METRICS
from discurso_odio.utils.plotting import setup_plot_style, get_label_names, get_class_palette
from discurso_odio.utils.seed import set_seed

settings = load_settings()
set_seed(settings['project']['seed'])
setup_plot_style()
LABELS = get_label_names()
REPORTS_FIGURES.mkdir(parents=True, exist_ok=True)
REPORTS_METRICS.mkdir(parents=True, exist_ok=True)
print('Proyecto:', settings['project']['name'])

from discurso_odio.evaluation.metrics import compute_all_metrics, bootstrap_ci, mcnemar_test, f1_score
from discurso_odio.evaluation.slices import slice_metrics, dialect_bias_probe
from discurso_odio.models.mbert import predict
from discurso_odio.paths import MODELS_DIR
import json
import joblib
from sklearn.metrics import f1_score as sk_f1



In [ ]:
test = load_split('test')
model_dir = MODELS_DIR / 'mbert-sv'
y_pred, y_prob = predict(test['texto_modelo'].tolist(), model_dir)
y_true = test['label'].values
metrics = compute_all_metrics(y_true, y_pred, y_prob)
print('F1-macro:', round(metrics['f1_macro'], 4))
print('Accuracy:', round(metrics['accuracy'], 4))
print('MAE ordinal:', round(metrics['ordinal_mae'], 4))
print(metrics['classification_report'])


In [ ]:
# Intervalos de confianza bootstrap
f1_mean, f1_lo, f1_hi = bootstrap_ci(y_true, y_pred, lambda yt, yp: sk_f1(yt, yp, average='macro'))
acc_mean, acc_lo, acc_hi = bootstrap_ci(y_true, y_pred, lambda yt, yp: (yt==yp).mean())
print(f'F1-macro bootstrap: {f1_mean:.4f} [{f1_lo:.4f}, {f1_hi:.4f}]')
print(f'Accuracy bootstrap: {acc_mean:.4f} [{acc_lo:.4f}, {acc_hi:.4f}]')
print(f'Objetivo F1>0.80: {"CUMPLE" if f1_mean >= 0.80 else "NO CUMPLE"}')


In [ ]:
# McNemar vs baselines y XLM-R (pareado sobre las mismas instancias de test)
from discurso_odio.evaluation.mcnemar import run_mcnemar_tests

mcnemar_df = run_mcnemar_tests(model_dir=model_dir, require_xlmr=True)
mcnemar_df.to_csv(REPORTS_METRICS / 'mcnemar_tests.csv', index=False)
mcnemar_df


In [ ]:
# Métricas por cortes (mBERT + XLM-R si hay predicciones auditadas)
import sys
from pathlib import Path
sys.path.insert(0, str(Path('scripts').resolve()))
from compute_slice_metrics import compute_all
from discurso_odio.evaluation.slices import dialect_bias_probe, export_dialect_bias
from discurso_odio.paths import TEST_CSV

slice_summary = compute_all(model_dir=model_dir, include_xlmr=True)
xlmr_status = slice_summary.attrs.get('xlmr_status', 'unknown')
from discurso_odio.evaluation.slices import export_slice_metrics
export_slice_metrics(
    slice_summary,
    REPORTS_METRICS,
    extra_metadata={'xlmr_status': xlmr_status},
    merge=True,
)
bias = dialect_bias_probe(
    test, y_true, y_pred,
    model='mbert',
    checkpoint=str(model_dir),
    split=TEST_CSV.name,
)
export_dialect_bias(bias, REPORTS_METRICS, extra_metadata={'xlmr_status': xlmr_status})
print('Sonda sesgo dialectal:')
print(bias[['group', 'n', 'fp', 'fpr', 'fpr_ci_lo', 'fpr_ci_hi']].to_string(index=False))
slice_summary


In [ ]:
# Matriz de confusión
from sklearn.metrics import ConfusionMatrixDisplay
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=LABELS, ax=ax, cmap='Blues')
plt.title('Matriz de confusión — mBERT test')
plt.tight_layout()
plt.savefig(REPORTS_FIGURES / '04_confusion_matrix.png')
plt.show()


In [ ]:
# Exportar métricas
export = {k: v for k, v in metrics.items() if not isinstance(v, (list, dict))}
export['f1_bootstrap_lo'] = f1_lo
export['f1_bootstrap_hi'] = f1_hi
with open(MODELS_DIR / 'mbert-sv' / 'test_metrics.json', 'w') as f:
    json.dump(export, f, indent=2)
pd.DataFrame([export]).to_csv(REPORTS_METRICS / 'mbert_test_metrics.csv', index=False)
print('Métricas exportadas.')
